In [10]:
# We will attempty to build a toy version of MIRA, following the paper's specifications and see if we can later tweak some of the 
# hyper parameters

![Screenshot](./views/graph1.png)

In [ ]:
# Let's start by importing the dataset as it is publicly available 

from huggingface_hub import snapshot_download

# since the train dataset is 7TB, we will only bring in two shards which is around 6GB

snapshot_download(
    "kyutai/rocket-science",
    repo_type='dataset',
    allow_patterns = ['test/index.json', 'test/dataset_00000.tar','test/dataset_00001.tar'], # the only 3 files we want (two shards)
    local_dir = './data/rocket'
)

# takes around 5 mins, you can give it a scroll

Fetching 3 files: 100%|██████████| 3/3 [05:40<00:00, 113.35s/it]


'/Users/achrafbayi/Desktop/MIRA/data/rocket'

In [ ]:
# now let's unpack the tar balls now

import os, tarfile

file1 = './data/rocket/test/dataset_00000.tar'
file2 = './data/rocket/test/dataset_00001.tar'
files = [file1, file2] # feel free to add as many shards as you need/can
dst = './data/rocket/test/unpacked'

os.makedirs(dst, exist_ok=True)

for file in files:
    with tarfile.open(file) as ball:
        ball.extractall(dst,filter='data')

# we now have the whole two shard dataset in disk ready to use

In [ ]:
# second step from the diagram above is to format it and feed it to the dinoV3-L feature extractor 
# (available on : https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m) request access asap to not have to sit
#  waiting for approval mid work

<img src="./views/graph2.png" width="600">

<b>Figure 3 Codec.</b> A frozen pretrained feature extractor (DINOv3-L) extracts per-frame features, mixed across several
intermediate layers, and a learned linear bottleneck downsamples them by 2 × 2 in space and 2× in time and projects
to a latent z with C channels. The decoder reverses these steps: a spatial upsampling restores the spatial resolution, a
causal spatio-temporal vision transformer reconstructs the video, and a temporal upsampling restores the frame rate.

(this is not figure 3 btw, but the figure 3 description explaines well what we need to do)

It is also quite important to understand that the Codec (encoder+decoder) are trained independently from the world model predictor and is frozen before predictor training time

In [ ]:
# as seen above a shared encoder is used and then concats on all of the 4 frames outputs in latent space to get the full visual 
# latent space/embeddings (to which we'll later add the other signals such as physics, key bindings, ect)
# let's hence import our encoder from hugging-face and see wassup

import torch
import torch.nn as nn
from transformers import AutoModel

device = 'mps' if torch.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'

# let's now define the encoder as a class that will implement all requirements from figure 3 and the graph which are : 
# (done individualy for all 4 files btw)
# 1) extract per frame features across several intermediate layers (exact mix is in table 9) using the DINOv3-L (torch.no_grad())
# 2) mix those embeddings by computing the mean, with no learnt bias (no NN involved)
# 3) linear bottle neck NN layer to down scale the obtained embeddings to our chosen channel number C (table 9) (Trained)
# 4) concat all 4 outputs

# from Table 9 of the implementation details, we get the exact codec training figures
# BatchSize is the number of frames we ingest

# moreover, Dino is a ViT style transformer, which hence splits our frames into patches and feeds them sequentially as tokens to the transformer
# and returns token embeddings

# moreover as for dimentionality, the paper mentions we need to preprocess our data into 288x512 format, which hence implies that the 
# that the number of tokens ingested and output-ed by the DINO ViT is of H/16 * W/16 = (288/16)*(512/16) = 18 * 32 ,
# hence the dimension of our VIT output is a tensor of shape (18, 32, 1024) and the multi-intermediate layers with the table 9 specification
# hence has us working with a (7, 18, 32, 1024) tensor that we scale down to a (18, 32, 1024) tensor through mixing
# and later, the t+1 merged with the other t frame which hence has us working with a (2, 18, 32, 1024) tensor is bottlenecked down
# through a linear neural net back to (9, 16, 32) embedding space (divided by 2x2 in space and 2x in time as per the paper)
# one interesting thing I might want to try out is using a neural net to mix rather than the mean (per the paper C = 32)

class Encoder(nn.Module):

    def __init__(self, layers=(11, 13, 15, 17, 19, 21, 23), C=32, BatchSize=40, H=288, W=512, n_embd=1024, is_training=False ): # ugly number of layers
        super().__init__()
        # btw 1024 is dino's encoding dim, not ours, ours is 1152
        self.dino = AutoModel.from_pretrained('facebook/dinov3-vitl16-pretrain-lvd1689m') # import DINO model
        self.dino.eval().requires_grad_(False) # making sure we don't backprop/optimize through dino (300M params)
        # btw for ViT-L, all embedding spaces have 1024 channels, hence : 
        self.layers = layers
        self.n_embd = n_embd
        self.C = C
        self.H = H
        self.W = W
        self.is_training=True
        k = len(layers)
        self.mix = nn.ParameterList([nn.Parameter(torch.full((H//16, W//16, n_embd),1/k)) for _ in range(k)]) # mixing into a single embedding space
        # here we explicitely blow up by one dimension the self mix to allow a granualrity-based mix, with different weights for 
        # each layer, we want to be able to learn if layer 11, 17 or 23 need more gain at a specific cell of the 18*32 
        # matrix of embedding vectors which hence allows the encoder to learn for example to differentiate foreground/position from 
        # texture if those sit at different layers of the DINO encoding
        # btw here the nn.Linear class was painful as it only supports in,out formats, hence we used nn.Parameters
        # we also start at 1/k which is the mean and is what is used by the paper to see if allowing optimization here improves loss

        # it gets us a (18, 32, 1024) shaped tensor x2 for the two frames we take as input and that we need to bottlneck

        # so we will be working with a (2,18,32,1024) that we want to scale down to (9, 16, 32)
        #(18, 32, 1024)
        # in similar fashion to what we did earlier
        self.bottleneck = nn.Conv3d(self.n_embd, self.C, kernel_size=2, stride=2)
         # times 8 since we basically divide by 1/2 * 1/2 * 1/2 the other dims, such that 8/8 = 1

    def _bottleneck(self, t):
        T, H, W, emb = t.shape
        output = t # we don't mind sharing pointers since the input is just inner state before we reach the latent space
        # I wish I could draw this, but here, we basically grab patches of 2x2x2 over the 3 first dimensions 
        # which lands us with 8 1024 element vectors per patch, which we each element wise multiply to a 1024 x C matrix 
        # to reduce those patch 'tokens' into the channel space we want, the challenging part being how to implement this mental model 
        # here, using .view to either target the right format right away or to try for first -1, 2, 2, 2, emb chunk the tensor doesn't work
        # this is highkey super complex to implement,
        # I had to this time give a look to the GH repo showing the implementation of the rae encoder (https://github.com/mira-wm/mira/blob/main/src/mira/codec/rae_encoder.py)
        # and they seem to use a Conv3D whose documentation is quite confusing but Claude nicely summarized for my specific use
        # Conv3D being a 3d convolutional layer, as we have encountered before in CNNs but this time allowing to control the step and size of our convolutions
        # here nonetheless, Conv3d expects to see the channels first which means we have to somehow properly reshuffle our input tensor 
        # to ensure the semantics fit and we don't just change the view and juggle everything out of place for the sake of .view

        # one helpful way to visualize how to move this I found is that if we imagine a 18 x 32 matrix holding 1024 vectors and try to get the 
        # [17,31, :]'th vector, it is the same thing as having 1024 [17,32] grids each holding a single scalar, and this time iterating over
        # depth as in : [:, 17, 31], we get the exact same result. and here we just need to put it in that format accounting for the extra dimention that is time
        # and since the first dimension here for Conv3d is already time, we are gucci I believe

        output = output.permute(3, 0, 1, 2) # here permutating dim 0 and 3 does the rotation I tried to describe which leaves us with 1024 18*32 tensors x Time
        output = self.bottleneck(output) # applying trainable convolutional network to merge the 3d - chunks into a single value
        output = output.permute (1,2,3,0) # permuting everything back into order
        return output

    def forward(self, frames):
        # assuming we get a frames tensor of shape [T, C, W, H] as input already pre-formated with H and W being respectively 288 and 512
        T, C, H ,W = frames.shape
        with torch.no_grad():
            features = self.dino(pixel_values=frames, output_hidden_states=True) # getting all hidden state
            # features.hidden_state is a tuple of [T, 581, 1024]
            selected_features = [features.hidden_states[i+1] for i in self.layers] # all the layers we want for layer mixing
            # an array of (T, 581, 1024) tensor with 5 extra tokens we need to clean off
            # that all sit at the beginning so easy to clean off
            selected_features = torch.stack([f[:,5:,:] for f in selected_features]) # stack to concat the array into a single tensor
            selected_features = selected_features.view(len(self.layers), T, self.H//16, self.W//16, self.n_embd) #viewing it accordingly
        # now we can mix em up
        mixed = torch.zeros_like(selected_features[0])
        for f,w in zip(selected_features, self.mix):
            mixed+= f * w #stacking up the dim=0 dim through additiong
        # we end up with a mixed in shape (T, H//16, W//16, self.n_embd)
        # we can now simply bottleneck it as we defined to get it to the latent space
        latent_space = self._bottleneck(mixed) # finally
        if self.is_training is True:
            return latent_space, selected_features # for training, in [F, T, H//16, W//16, n_embd]
        else:
            return latent_space

<img src="./views/decoderspec.png" width="1100px">
<img src = "./views/DecoderHyperparams.png" width = "1100px">
<img src = "./views/transformer.webp" width = "400px">

In [3]:
# now that we have the encoder, if we want to train, we need to build the decoder and later pre-process the data and I guess import 
# some val shards too to really see how good of a loss we will be able to get 

import math
# we need to build our own implementation of a space-time ViT whose hyperparameters are specified in Table 9


class ViT(nn.Module): # the paper specifies in table 9 a depth of 28 so 28 (attention->mlp) blocks stacked, let's just specify the blocks
   def __init__(self, depth=28, T=20, H=18, W=32, n_embd=1152):
      super().__init__()
      self.blocks = nn.ModuleList([Block() for _ in range(depth)]) # instantiating depth-long array of blocks
      self.positional_embeddings = nn.Parameter(torch.randn((T,H,W,n_embd))*0.02) # learned positional embeddings, applied once  before as specified in the transformer architecture
      #scaling down the positional embeddings as seen in GPT-2 

   def forward(self, x):
      # once again, with (T, H, W, n_embd) shaped input, we'll sequentially apply the blocks
      out = x + self.positional_embeddings
      for block in self.blocks:
         out = block(out)
      return out


class Block(nn.Module):
   def __init__(self):
      super().__init__()
      self.attention = Attention() # we hardcoded the paper's default values correctley so no need to specify anything for now unless we would later like to change a hyperparameter
      self.mlp = MLP()

   def forward(self,x): # here again, input should be of format (T,H,W,n_embd)
      out = self.attention(x)
      out = self.mlp(out) 
      # btw all pre-norm layer normalization and residual connections are implemented within the attention and mlp blocks so no need to do anything here
      return out



class MLP(nn.Module): 
   def __init__(self, n_embd=1152, mlp_mult=4):
         super().__init__()
         # input comes as (T,H,W,n_embd) and we work with a 4x multiplier on n_embd
         self.n_embd = n_embd
         self.mlp_mult = mlp_mult
         self.ln = nn.LayerNorm(n_embd) # pre-norm ln learned gammas and betas
         self.layer1 = nn.Linear(n_embd, n_embd*mlp_mult) # scaling up to the MLP's dim
         self.non_linearity = nn.GELU() # non-linearity
         self.layer2 = nn.Linear(n_embd*mlp_mult, n_embd) # scaling back down to the embd's dim

   def forward(self, x):
      out = self.ln(x) # pre-norm
      out = self.layer1(out)
      out = self.non_linearity(out) #GELU
      out = self.layer2(out)
      out = out + x # residual connection
      return out

# works with (T, H, W n_embd) inputs and returns (T, H, W, n_embd) with space and time attention applied sequentially
class Attention(nn.Module):
   def __init__(self, n_head=16, n_embd=1152):
      super().__init__()
      self.n_embd=n_embd
      self.head_dim = n_embd//n_head
      self.space_heads = nn.ModuleList([SpaceAttentionHead() for _ in range(n_head)])
      self.time_heads = nn.ModuleList([TimeAttentionHead() for _ in range(n_head)])
      self.space_ln = nn.LayerNorm(n_embd)
      self.time_ln = nn.LayerNorm(n_embd) # since we'll work with the output of the 1st space attention layer
      self.space_mixing = nn.Linear(n_embd, n_embd)
      self.time_mixing = nn.Linear(n_embd, n_embd) # linear layers we'll use to mix the concatenations 

   def forward(self, x):
      # here x is of dim (T, H, W, n_embd)
      # we need first to apply spatial attention to all frames and then time attention too
      T, H, W, emb = x.shape
      out = self.space_ln(x) # pre-norm layer norm
      out = torch.concat([h(out) for h in self.space_heads], dim=-1).reshape(T,H,W,self.n_embd) # applying space attention, we also let a residual connection in
      # reshaping into 2d output for us to have coherent space and time attention head code and to be consistent
      out = self.space_mixing(out) #mixing
      out = out + x #residual connection for space attention (2d)
      newx = out # output of the space attention is the new input for the time attention
      out = self.time_ln(newx) # 2nd pre-norm layernorm
      out = torch.concat([h(out) for h in self.time_heads], dim=-1).reshape(T,H,W,self.n_embd) # stacking the columns
      out = self.time_mixing(out)
      out = out + newx # residual connection for time attention
      return out # simple this time


class SpaceAttentionHead(nn.Module): # not a join space-time attention, we are applying two reqular attention QK sequentially rather than a 3 dimensional one
   def __init__(self, H=18, W=32 , n_embd=1152, n_head=16, T=20):
      super().__init__()
      self.T = T
      self.H = H
      self.W = W
      self.n_embd = n_embd
      head_dim = n_embd//n_head
      #(T, H, W, n_embd) here is what is fed to us, we want to sequentially apply space attention (self) and later time attention (causal)
      # we consume frame by frame, hence working in ( H, W, n_embd) work space
      # we'll hence move into (H*W, n_embd) token space as specified in the ViT An image is a 16x16 word space as our token sequence
      # through a .reshape() on the input tensor hence we'll just need a (H*W, n_embd)-long learned matrix for Q, K and V
      self.wQ = nn.Linear(n_embd, head_dim)
      self.wK = nn.Linear(n_embd, head_dim) # @ matmul dot product to get the QK (H*W, H*W) shape
      self.wV = nn.Linear(n_embd, head_dim) # values to be multiplied to the layernormed, softmaxed attention through matmul again
      # all learned

      # so the outputed Q,K,V will be of shape :
      # Q.shape = (H*W, n_embd)
      # K.shape = (H*W, n_embd) we will .transpose to run the dot products properly
      # QK.shape = (H*W, H*W), good
      # V.shape = (H*W, n_embd//16)

      # finally the layer normalization learned gamma and betas 
      self.q_ln = nn.LayerNorm(head_dim)
      self.k_ln = nn.LayerNorm(head_dim)
   def forward(self, x):
      # x is of shape (H,W,n_embd)

      x = x.reshape(self.T, self.H*self.W, self.n_embd) # flatening the tokens (576 tokens)
      Q = self.wQ(x)
      K = self.wK(x)
      V = self.wV(x)
      # we now apply our layer norms real quick 
      Q = self.q_ln(Q)
      K = self.k_ln(K)
      dnorm = 1/(math.sqrt(K.shape[-1])) #dimentionality of K
      QK = Q @ K.transpose(-2,-1) # using permute to transpose to run the dot products
      QKnormed = QK * dnorm
      softmaxedQK = torch.softmax(QKnormed, dim=-1) # on the columns
      out = softmaxedQK @ V
      return out

class TimeAttentionHead(nn.Module):
   def __init__(self, n_head=16, n_embd=1152, T=20, H=18, W=32):
      super().__init__()
       # so here, again, we'll be working with (T, H, W, n_embd) tensor
      self.emb_head = n_embd//n_head
      self.wQ = nn.Linear(n_embd, self.emb_head)
      self.wK = nn.Linear(n_embd, self.emb_head)
      self.wV = nn.Linear(n_embd, self.emb_head)
      self.dk = 1/(math.sqrt(self.emb_head)) # dimensionality of the keys
      # we also need our layer norm layers for the QK normalization applies in layer norm
      self.ln_q= nn.LayerNorm(self.emb_head)
      self.ln_k= nn.LayerNorm(self.emb_head)

   


   def forward(self, x):
      # we'll first squash it down to 
      # (T, HxW, n_embd) tokens, and we'll hence want to compute attention this time on all the previous
      # mappings of that token in the other frame, to make that easier, we may want to try to reshape the input
      # to something like (H*W, T, n_embd) which can be seen as a matrix who'se rows represnt a specific position token on the image
      # and column value its time embedding, hence we may want to apply attention over itself and all previous ones which would indeed
      # be a lower-triangular matrix
      # let's first

      T,H,W,n_embd = x.shape

      x = x.reshape(T, H*W, n_embd) # automatically picks the right column to H which is W so it works here natively
      # now we want to permute
      x = x.permute(1,0,2) # going to (H*W, T, n_embd) (i.e, [frame1, frame2, frame3,....]) each frame being made of N*W tokens of embedding n_embd
      # and now we may want to generate our 
      Q = self.wQ(x)
      K= self.wK(x)
      V = self.wV(x)
      # Q.shape, K.shape, V.shape = (H*W, T, head_emb)
      # QK norm
      Q = self.ln_q(Q)
      K = self.ln_k(K)
      QK = Q @ K.permute(0, 2, 1) #pairwise matmul
      QK = QK * self.dk
      # now we need to mask
      mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device = x.device)) # of same shape as QK
      QK = QK.masked_fill(~mask, float('-inf')) # btw ~ is the bitwise not operator meaning on NOT True (bool True) it replaced by -inf
      out = torch.softmax(QK, dim=-1) # on the columns
      out = out @ V #(H*W, T, head_emb) shape through tensor-wise mat-mul
      out = out.permute(1,0,2) # going back to frames first
      return out

In [4]:
# now that we have all subclasses, we can put them together in the Decoder and handle the upscalings/unpatchifications there
class Decoder(nn.Module):
   def __init__(self, T=20, H = 9, W= 16, C=32, n_embd=1152):
      super().__init__()
      # this time, input is of the shape (T//2, H//2, W//2, C) (unchanged from the encoder) and we want to bring it back to 2x the dimn on T,W,W and n_embd on C
      # the first step is the unpatchification which is similar to the patchification we've done earlier,
      # we need to understand that we want to scale back C -> 1152, W//2 -> W, H//2 -> H and T//2 -> T 
      # here is how we proceed to do so, scaling C is trivial, we howevere here want to scale it to 4*1024 = 4096 through good ole matrix multiplication
      # which will hand us back (T//2,H//2,W//2,4096), and we want to view those 4608 vectors as 2 x 2 matrices of each 1152 elements which will scale 
      # H and W in both dimension by 2
      self.T = T
      self.H = H
      self.W = W
      self.C = C
      self.n_embd = n_embd
      self.spatial_upsample = nn.Linear(C, n_embd*4)
      # ourput will be in the following format : (T, 2H, 2W, n_embd) (i.e 10, 18, 32 , 1152)
      self.vit = ViT() # default hardcoded values are correct for the paper's hyperparams for now
      self.time_upsample = nn.Linear(n_embd, n_embd*2) #doubling, shared
      self.token_scale_conv = nn.ConvTranspose2d(n_embd, 3, kernel_size=16, stride=16) # as mentioned in the paper, each token in the 228*512 grid embedds a 16x16 pixel patch in the 
      # real image, and each pixel embedds RGB, i.e 3 uint8 values, hence why the 16*16*3 final embedding, we use a convolutional network to unsplit the 768 embd into 16x16 tokens


   def _spatial_upsample(self, t):
      out = t
      out = self.spatial_upsample(out) # growing back the last dimension
      # doing the view operation we mentioned above to have the H x W matrix go from storing 4608 vectors to 2 x 2 sub matrices 
      # of 1024 elements each in its cells
      out = out.view(self.T, self.H, self.W, 2, 2, self.n_embd)
      # now here, the permuts -> reshape work to un-nest that 2x2 can get pretty trick and was hard to understand at first glance seeing how 
      # shitty the PyTorch documentation can sometimes be ( this one sucks : https://docs.pytorch.org/docs/2.14/generated/torch.permute.html)
      # but the one for reshape is actually useful here (https://docs.pytorch.org/docs/2.14/generated/torch.reshape.html)
      # what we will try to achieve is to just permute (i.e swap the stride layout according to Claude) to get a tensor of shape : 
      # (T, ((H, 2), (2, W)), n_embd) in which we will be working on the sub tensors : ((H, 2), (2, W)), visualizing those as a H x 2 matrix 
      # of 2 x W sub-matrices helps to see how the shape is conserved if ever layed down on paper, and reshape here will simply allow us to in the
      # following order : 
      # 1) : turn the 2 x W matrix into a 2W long vector (which is the original goal of 2x-ing W)
      # 2): now that we have a H x 2 matrix of (2W) long vectors, we will also reshape (i.e append the 2nd col downwards to the 1st)
      #     the H x 2 into a 2H long column of 2W long rows which is exactly a 2H x 2W matrix of 1152 scalar vectors which is 
      #     what we initially seeked to do, let's implement that :
      out = torch.permute(out, (0,1,3,2,4,5)) # doing the permutations to get a ((H, 2), (W, 2)) in the middle
      out = out.reshape(self.T, self.H, 2, self.W*2, self.n_embd) # one cool thing about torch.reshape is that we don't need to isolate the 
      # sub matrix we want to flatten, we just specify the new shape, and it AUTOMATICALLY grabs the elements from the axis to the right of
      # the axis we want to 'flatten/reshape' to do so which is why we got it into that shape through permute
      # now we just do step 2 and we're gucci on all dims but T
      out = out.reshape(self.T, self.H*2, self.W*2, self.n_embd)

      return out
   
   def _time_upsample(self, x):
      # input is of dim (20, 18, 32, 1152), we want to individually upscale, each frame into anothe one, we could do this linearly by building another
      # list of 20 frames from the 10 frames and appending it to the end of the existing one, but let's try to do this in a parallel way for the sake of efficiency
      # what is we tried to lay out the frames linearly by stacking them one on top of another (maybe through .reshape(1,-1,32,1152))
      # and then applied a (1152, 2304) matrix multiplication neural net to double the scale of each embedding, 
      # and then we cut those embeddings back to 1152, and the trimmed part becomes the new frames
      # to do so efficiently, since we know that reshape automatically grabs elements from the right side, we may first want to hit our new tensor
      # with the .reshape(1,-1,32,2,1152) which will split it into two from the grown embeddings which is what we want
      # and we then may want to concat those at the bottom, maybe using a squeeze operation with the shape :
      # .squeeze(1,-1,32,1152), and then later we may just want to reshape back to 40 and 32 with reshape .reshape(-1, 18, 32, 1152) which 
      # should yield 40 for the 1st dim as we doubled the elements pairwise
      # all done parallely which was our goal, let's try to implement that 
      T, H, W, n_embd = x.shape
      out = x.reshape(T,-1,W,n_embd) # stacking all frames along the row dimension, we hence stack all frames on height which yeilds (18x20, 32, 1152), good
      out = self.time_upsample(out) # doubling all embeddings, through shared matmul
      out = out.view(T,H,W,2,n_embd) # spliting the new embeddings into two (20, 18 32, [1152, 1152]) without touching memory order
      # where we now basically have the 2 dimensionality indexing between the parent frames and the children frames (21 and up)
      out = out.permute (0, 3, 1, 2, 4) # we need to permute to be able to properly use reshape on the height dim (i.e bring the 
      #[1152,1152] tensors to the right side of the time frame of the matrix so that we get (20, 2, 18, 2, 32, 1152)
      out = out.reshape(2*T, H, W, n_embd) # we want to time dimension to eat those tuples in 2
      return out # those are always so tricky to execute, idk how many times I re-wrote this block to get to the semantically correct one

   def forward(self, x):
      #x should be of shape (20, 9, 16, 32), we will first upscale it as seen in the _spatial_upsample
      out = self._spatial_upsample(x)
      # out should now be of (20, 18, 32, 1152) shape which we will put through our built ViT
      out = self.vit(out)
      # now space and time attentioned, should be still of (20, 18, 32, 1152) shape, we now just need to upsample the time dimension and we should be done
      # here we need to watch out for the framing of the tranformation to not mix everything up, let's say we have 10 frames indexed as such :
      # [0,...,9], the transfo takes us to [0,..,19] and frame 19 is built only from frame 9's data, and so on for frame 18 and 8 ect..
      # this is a bijective/1:1 learned transformation that we'll now implement
      out = self._time_upsample(out)
      # we are now back in the right time dimensionality with [20, 18, 32, 1152], we just need to scale back up height and width from 18 x 32 
      # to 288 x 512.
      # in the decoder this was done through linearly scaling down the outputs of the 7 layers (through a regular neural net) and mixing them in 
      # a learned way for our specific case or just doing a mean in the paper's specification
      # here though to upscale, we don't have the need to do it in a learned way, since our 16x16 tokens (in the 18x32 referentials) are 
      # already convolutions of the larger 16x16 tokens in the 288x512 one, we just need to split accordingly our embeddings into those tokens
      # all the data needed already holds into the 16x16 tokens at (18x52) to rebuild the ones at the (288x512) scale
      # the only thing we actually need to do is downscale our 1152 embeddings to 768 as specified in the __init__ for that linear layer above
      out = self.token_scale_conv(out.permute(0,3,1,2)) # moving down to 768 and then reverse conv our way to 16x16 chunks bilt on those 768 embeddigns
      # btw we need to permute here as convTranspose2d expects the channel to be first [T, 3, H,W]
      
      return out #finally, at [T, 3, H, W] format, we should be more than good

<img src="./views/imagedocumentation.png" width="900px">
<img src ="./views/mean_normalization.avif" width="300px">

In [ ]:
# we are virtually done with the codec, the encoder and decoder being end-to-end complete, the decoder being the hardset part is behind us
# we just need two small pre_processing and post_processing classes to take the images from 229

# as specified in the documentation of the dataset, the input frames are of format 1280x720, hence for our frame batch size of T
# we'll be working with inputs of the shape [P, T, 3, 720, 1280] # 3 for rgb
# P being the players and T being the number of frames
# I'd like to off-load the P and T handling to the pre-pre-procesing that will happen before training/inference and we will hence assume that 
# the input here will be of shape [T, 3, 720, 1280] T being arbitrarly chosen

from transformers import AutoImageProcessor

class pre_processor(nn.Module): # parameter less, takes [T, 3, 1280, 720] down dimension wise on H and W
    def __init__(self):
        super().__init__()
        #p = AutoImageProcessor.from_pretrained('facebook/dinov3-vitl16-pretrain-lvd1689m')
        #mean = p.image_mean is (0.485, 0.456, 0.406)
        #std = p.image_std is (0.229, 0.224, 0.225)
        # here it is ok to hardcode them since the weights are frozen
        self.mean = torch.tensor([0.485, 0.456, 0.406]).reshape(3,1,1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).reshape(3,1,1) # we reshape for the broadcasting to happen as expected
        # since broadcasting aligns from the right, we want 3 to land on the 3 in [T,3,H,W]


    def forward(self, x): # single function call on [T, 3, 1280, 720]-shaped x
        out = torch.nn.functional.interpolate(x.float(), size=(288, 512), mode="bilinear", antialias=True)# interpolate expects channels at 1 so we are already good on that
        # output is now of shape [T, 3, 720, 1280]
        # bilinear since (1280/720) == (512/288) (i.e same dimension, no cropping/shearing/squeezing is required)
        # antialias to smoothen down jagged lines through avergaing s/o Intro to Computer Graphics!
        # btw [T,C,H,W] is the shape DINO expects so no need to permute back

        # btw we need it to be float for later normalization and for interpolate 
        # we now want to normalize as specified by DINO for it to be consumed by it
        out = out * (1/255) # normalizing to get 0,1 values
        # we also want to apply mean normalization on the DINO mean and std 
        # to do so we need the model's config.json

        out = (out - self.mean.to(x.device))/self.std.to(x.device) # according to broadcasting rules, this should work just fine
        return out # we return a [T, 3, 288, 512] shaped tensor

# to take back the 
# [T, 288, 512, 3] shaped tensor into viewable images of the same shape as the [T, 3, 720, 1280] input
# btw the post-processor is only useful for inference as during training, we compute loss on the 288x512 images
class post_processor(nn.Module): # also parameter less
    def __init__(self):
        super().__init__()
        self.mean = torch.tensor([0.485, 0.456, 0.406]).reshape(3,1,1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).reshape(3,1,1)
        # same as above

    def forward(self, x): # x is the decoder's output of shape [T, 3, 288, 512]
        # let's first reverse mean-std normalization
        out = x*self.std.to(x.device)
        out = out + self.mean.to(x.device)
        out = out.clamp(0,1)
        out = torch.nn.functional.interpolate(out, size=(720, 1280), mode="bilinear") # same as above, just to scale up this time
        out = out * 255
        out = out.round() 
        out = out.to(torch.uint8) # rgb's dtype
        # we are now in [T, 3, 720, 1280]
        return out

<img src="./views/loss_details.png" width="1000px">
<img src="./views/loss_metrics.png" width="1000px">
<img src="./views/optimizer_metrics.png" width="1000px">
<img src="./views/cosine_schedule.png" width="500px">

In [ ]:
# now that we have the 4 parts, putting the codec together, and giving it a train(), infer() functions as well as loss

import torch.nn.functional as F
import lpips

class Codec(nn.Module):
    def __init__(self, max_lr=2e-4, b1b2 = (0.9,0.95), wdecay = 0, warmup_steps=1000, min_lr = 1e-6, total_steps=249000, batch_sizes = ):
        # out sub-blocks
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()
        self.pre_processor = pre_processor()
        self.post_processor = post_processor()

        #parameter list
        self.parameter_list = list(self.decoder.parameters()) + [p for p in self.encoder.parameters() if p.requires_grad]
        # since some elements of the encoder don't require grad such as the dinoV3-L

        # optimizer
        self.optimizer = torch.optim.AdamW(self.parameter_list, betas=b1b2, weight_decay=wdecay) # eps is default, paper specifies AdamW but not the lr schedule which we will assume cosine like in GPT-2
        self.warmup_steps=warmup_steps
        self.total_steps=total_steps
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.lpips = lpips.LPIPS(net="vgg").eval().requires_grad_(False) # as mentioned in the paper (blackbox implementation lowkey, I did NOT read that paper yet)
        self.mean = torch.tensor([0.485, 0.456, 0.406]).reshape(3,1,1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).reshape(3,1,1)
        # hardcoded means and std from ImageNet

    # we need to implement a couple inner functions

    def _get_cosine_lr(self, step): # at step : step
            if step<=self.warmup_steps:
                # linear warmup function
                return step * ((self.max_lr - self.min_lr)/self.warmup_steps) + self.min_lr
                # f(0) = min_lr
                # f(100) = max_lr
                # f(x) = ax+b
                # -> b = min_lr
                # warmup*a + min_lr = max_lr <=> a = (max_lr - min_lr)/warmup
                # f(step) = (step * (maxlr-minlr)/warmup) +minlr # linear warmup function
            elif step>self.total_steps: # should not happen but we still put it
                return self.min_lr
            else:
                decay_ratio = (step-self.warmup_steps) / (self.total_steps-self.warmup_steps) # as computed in the GPT-2 rebuild
                assert 0<=decay_ratio<=1
                coeff = 0.5*(1.0+math.cos(math.pi*decay_ratio))
                return self.min_lr + coeff * (self.max_lr-self.min_lr) # all from the GPT-2 buildout btw

    def _imagenet_unnormalization(self, x):
        # x of shape [T, C, H, W] with C = 3 (R,G,B)
        norm = x*self.std.to(x.device)
        norm = norm + self.mean.to(x.device)
        norm = norm*2
        norm = norm -1 # to shift from [0,1] to [-1, 1] as expected by lpips
        return norm
                
    def loss(self, x, y, intermediate_layers):
        # per the paper, loss seems to have 3 components, a regular L1 component, a LPIPS component, and a re-encode through the DINOV3-L
        # loss component that re-encodes the output and computes a L1 on different intermediate layers
        # loss is averaged over all components in all dimensions and the returned as a single valye on which we'll call backwards
        # x input is of shape (B*T, C, H, W) # B being the batch size, we put all the frames on the T scale when batched
        # y is of the shape (B*T, C, H, W) and si the output of the decoder without post_processing applied

        #! BTW, the two non-l1 components only ever get computed on 25% of the frames in a batch,

        N = x.shape[0] # number of frames passed in the batch
        mask = torch.rand(N, device=x.device) < 0.25 # we'll use this to mask x and y and only get 25% of the frames for the non-l1 component loss element

        # let's first compute the l1 loss component

        l1_component = F.l1_loss(x,y)

        # now we need the intermediate layer's output, to get that we will implement three changed to the decoder :
        # 1) goal will be to cache the intermediate layers that we used for inference (the exact layers are not specified but this 
        # decision makes sense for the sake of efficiency)
        # 2) we'll add a is_training boolean flag to the encoder (default False) such that it only stores those values for training and not for inference
        # 3) obv make sure to pass them into loss so that we don't have to re-compute them
        # btw they're of format : [F, T, H//16, W//16, n_embd] or here , [7, T, 18, 32, 1024]
        # i.e before the bottlneck that halves T, H, W
        # btw it is not l1 loss that is applied to them but mean of squared differences
        # and we also need to re-encode the output's y into the encoder's feature extractor and grab the intermediate dimensions 
        # in the same way we've done at inference time
        if mask.any():
            il_output = self.encoder.dino(pixel_values=y[mask], output_hidden_states=True) # taking in the mask
            il_output = [il_output.hidden_states[i+1] for i in self.encoder.layers] # it is good that in python, class member access if straighforward
            il_output = torch.stack([f[:,5:,:] for f in il_output]) # code from the forward of encoder btw
            il_output = il_output.view(intermediate_layers[:,mask].shape) # to be sure we're in the same shape

            intermediate_msd = F.mse_loss(il_output, intermediate_layers[:,mask]) # taking in the mask
         
        # now we need to implement the lpips component

            lpips_component = self.lpips(self._imagenet_unnormalization(x[mask]),self._imagenet_unnormalization(y[mask])).mean() # also masked on 25% of frames in the batch

        # gains
            eps = 1e-4
            w = self.decoder.token_scale_conv.weight
            g_rec = torch.autograd.grad(l1_component, w, retain_graph=True)[0].norm()
            g_msd = torch.autograd.grad(intermediate_msd, w, retain_graph=True)[0].norm()
            g_lp  = torch.autograd.grad(lpips_component, w, retain_graph=True)[0].norm()
            lambda_msd   = (g_rec / (g_msd + eps)).detach()
            lambda_lpips = (g_rec / (g_lp + eps)).detach()

            loss = l1_component + lambda_msd*intermediate_msd + lambda_lpips*lpips_component # final loss 

        else: #scenario of an mask that yeilds an empty tensor (errors)
            loss = l1_component

        return loss

    def forward(self, x): # x of shape [T,C,H,W]
        out = self.pre_processor(x)
        out = self.encoder(out)
        out = self.decoder(out)
        return out # lowkey irelevant for now, maybe I should change it when the world model will be plugged into encode and decode functions

    def train_loop():
        # where the training happenss




# btw we also need a DataLoader class that we will use to pre-process the data

    
        
        

